In [1]:
# ============================================================
# Re-fit UWB bias and thermal model on dist_raw_m
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
np.set_printoptions(precision=4, suppress=True)

# ------------------------------------------------------------
# 1. Load
# ------------------------------------------------------------
FILES = {
    1: "dataset_raw_static_square_r1_10min.csv",
    2: "dataset_raw_static_square_r2_10min.csv",
    3: "dataset_raw_static_square_r3_10min.csv",
    4: "dataset_raw_static_square_r4_10min.csv",
}

GT_PAIR = {
    (1, 2): 2.0000, (1, 3): 2.8284, (1, 4): 2.0000,
    (2, 1): 2.0000, (2, 3): 2.0000, (2, 4): 2.8284,
    (3, 1): 2.8284, (3, 2): 2.0000, (3, 4): 2.0000,
    (4, 1): 2.0000, (4, 2): 2.8284, (4, 3): 2.0000,
}

frames = []
for rb, fname in FILES.items():
    p = Path(fname)
    if not p.exists():
        print(f"MISSING: {fname}")
        continue
    d = pd.read_csv(p)
    d["source_robot"] = rb
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(df)} rows")

df = df[df["status"] == 1].copy()
df = df.dropna(subset=["dist_raw_m", "dist_cfo_m"]).reset_index(drop=True)
print(f"After status==1 filter: {len(df)} rows")

# ------------------------------------------------------------
# 2. Per-link mean bias (raw vs cfo)
# ------------------------------------------------------------
print("\n" + "=" * 90)
print(" PER-LINK MEAN BIAS  (raw vs cfo)")
print("=" * 90)
print(f"{'link':>6s} | {'n':>5s} | {'GT':>7s} | "
      f"{'raw_mean':>10s} {'raw_bias':>10s} | "
      f"{'cfo_mean':>10s} {'cfo_bias':>10s}")
print("-" * 90)

link_stats = {}
for (i, j), gt in GT_PAIR.items():
    mask = (df["init_id"] == i) & (df["resp_id"] == j)
    g = df[mask]
    if len(g) < 100:
        continue
    raw_mean = float(g["dist_raw_m"].mean())
    cfo_mean = float(g["dist_cfo_m"].mean())
    link_stats[(i, j)] = {
        "raw_mean": raw_mean,
        "cfo_mean": cfo_mean,
        "raw_bias": raw_mean - gt,
        "cfo_bias": cfo_mean - gt,
        "n": len(g),
    }
    print(f"{i}->{j}   | {len(g):>5d} | {gt:>7.4f} | "
          f"{raw_mean:>10.4f} {raw_mean - gt:>+10.4f} | "
          f"{cfo_mean:>10.4f} {cfo_mean - gt:>+10.4f}")

# ------------------------------------------------------------
# 3. Fit A + B + c model  (with sum(A) = sum(B) = 0)
# ------------------------------------------------------------
def fit_ABc(bias_dict):
    """bias_dict: {(i,j): bias_value}. Returns A, B, c, rmse, resid."""
    rows, y = [], []
    for (i, j), b in bias_dict.items():
        row = np.zeros(9)          # A1..A4, B1..B4, c
        row[i - 1] = 1.0
        row[4 + j - 1] = 1.0
        row[8] = 1.0
        rows.append(row)
        y.append(b)
    A_mat = np.array(rows)
    y     = np.array(y)

    # gauge: sum(A)=0, sum(B)=0
    g1 = np.zeros(9); g1[0:4] = 1.0
    g2 = np.zeros(9); g2[4:8] = 1.0
    A_aug = np.vstack([A_mat, g1, g2])
    y_aug = np.concatenate([y, [0.0, 0.0]])

    coef, *_ = np.linalg.lstsq(A_aug, y_aug, rcond=None)
    A_vals = coef[0:4]
    B_vals = coef[4:8]
    c_val  = coef[8]
    pred   = A_mat @ coef
    resid  = y - pred
    return A_vals, B_vals, c_val, float(np.sqrt(np.mean(resid ** 2))), resid

raw_bias_dict = {k: v["raw_bias"] for k, v in link_stats.items()}
cfo_bias_dict = {k: v["cfo_bias"] for k, v in link_stats.items()}

print("\n" + "=" * 90)
print(" STATIC BIAS MODEL  d_meas = d_true + A_i + B_j + c")
print("=" * 90)

for label, bias_dict in [("RAW", raw_bias_dict), ("CFO", cfo_bias_dict)]:
    A_vals, B_vals, c_val, rmse, resid = fit_ABc(bias_dict)
    print(f"\n[{label}]   c = {c_val:+.4f} m   RMSE = {rmse*100:.2f} cm")
    print(f"   A = {['%+.4f' % a for a in A_vals]}")
    print(f"   B = {['%+.4f' % b for b in B_vals]}")
    print(f"   per-link residuals:")
    for k, r in zip(bias_dict.keys(), resid):
        print(f"     {k[0]}->{k[1]}: {r*100:+7.2f} cm")

# ------------------------------------------------------------
# 4. Per-link bias matrix (A_i + B_j + c)
# ------------------------------------------------------------
print("\n" + "=" * 90)
print(" IMPLIED DIRECTED-LINK BIAS MATRIX   A_i + B_j + c")
print("=" * 90)

for label, bias_dict in [("RAW", raw_bias_dict), ("CFO", cfo_bias_dict)]:
    A_vals, B_vals, c_val, _, _ = fit_ABc(bias_dict)
    mat = np.zeros((4, 4))
    for i in range(4):
        for j in range(4):
            if i == j:
                mat[i, j] = 0.0
            else:
                mat[i, j] = A_vals[i] + B_vals[j] + c_val
    print(f"\n[{label}]  (rows = initiator, cols = responder)")
    print("       " + "".join(f"  →R{j+1:<8d}" for j in range(4)))
    for i in range(4):
        row_str = f"  R{i+1} |"
        for j in range(4):
            row_str += f"  {mat[i,j]:+9.4f}"
        print(row_str)

# ------------------------------------------------------------
# 5. Thermal model fit  (bias_resid = b0 + b1*ΔTavg + b2*ΔTdiff)
# ------------------------------------------------------------
def fit_thermal(df, dist_col, bias_dict):
    """Fit per-link thermal coefficients."""
    out = {}
    for (i, j), gt in GT_PAIR.items():
        mask = (df["init_id"] == i) & (df["resp_id"] == j)
        g    = df[mask]
        if len(g) < 100:
            continue

        bias = g[dist_col].values - gt - bias_dict[(i, j)]

        Ti = g["temp_uwb_init"].values
        Tj = g["temp_uwb_resp"].values

        T_avg  = 0.5 * (Ti + Tj)
        T_diff = Ti - Tj

        T_avg_c  = T_avg  - T_avg.mean()
        T_diff_c = T_diff - T_diff.mean()

        X = np.column_stack([np.ones(len(bias)), T_avg_c, T_diff_c])
        beta, *_ = np.linalg.lstsq(X, bias, rcond=None)

        pred  = X @ beta
        resid = bias - pred
        ss_res = float(np.sum(resid ** 2))
        ss_tot = float(np.sum((bias - bias.mean()) ** 2))
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")

        out[(i, j)] = {
            "b0": beta[0], "b1": beta[1], "b2": beta[2],
            "r2": r2, "rmse_cm": float(np.std(resid) * 100),
            "T_avg_mean":  float(T_avg.mean()),
            "T_diff_mean": float(T_diff.mean()),
        }
    return out

print("\n" + "=" * 90)
print(" THERMAL MODEL FIT  per directed link")
print("=" * 90)

raw_thermal = fit_thermal(df, "dist_raw_m", raw_bias_dict)
cfo_thermal = fit_thermal(df, "dist_cfo_m", cfo_bias_dict)

print(f"\n{'link':>6s} | {'RAW b1':>10s} {'RAW b2':>10s} {'RAW R²':>8s} "
      f"| {'CFO b1':>10s} {'CFO b2':>10s} {'CFO R²':>8s}")
print("-" * 90)
for k in sorted(GT_PAIR.keys()):
    r = raw_thermal.get(k)
    c = cfo_thermal.get(k)
    if r is None or c is None: continue
    print(f"{k[0]}->{k[1]}   | "
          f"{r['b1']*100:>+10.3f} {r['b2']*100:>+10.3f} {r['r2']:>8.4f} | "
          f"{c['b1']*100:>+10.3f} {c['b2']*100:>+10.3f} {c['r2']:>8.4f}")
print("  (b1, b2 in cm/°C)")

# ------------------------------------------------------------
# 6. Compare with existing UWBPreprocessor values
# ------------------------------------------------------------
print("\n" + "=" * 90)
print(" COMPARISON WITH EXISTING UWBPreprocessor TABLES")
print("=" * 90)

# Existing tables (from UWBPreprocessor.h)
EXISTING_BIAS = np.array([
    [  0.0000,  19.2418,  21.6532,  19.5407],
    [ 58.5328,   0.0000,  41.0874,  39.3955],
    [ 55.7571,  36.0566,   0.0000,  37.0840],
    [ 57.6726,  38.2033,  40.8134,   0.0000]
])

EXISTING_B1 = np.array([
    [  0.0000, -0.2466, -0.1787, -0.2430],
    [ +0.2990,  0.0000, +0.1227, -0.0005],
    [ +0.0774, -0.1263,  0.0000, -0.1332],
    [ +0.2767, -0.0087, +0.1091,  0.0000]
])

EXISTING_B2 = np.array([
    [  0.0000, -0.1664, -0.1072, -0.1723],
    [ -0.4949,  0.0000, -0.0170, -0.0128],
    [ -0.3601, -0.0517,  0.0000, -0.0709],
    [ -0.4920, -0.0094, -0.0225,  0.0000]
])

# New values from RAW fit
A_raw, B_raw, c_raw, _, _ = fit_ABc(raw_bias_dict)
NEW_BIAS_RAW = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        if i != j:
            NEW_BIAS_RAW[i, j] = A_raw[i] + B_raw[j] + c_raw

NEW_B1_RAW = np.zeros((4, 4))
NEW_B2_RAW = np.zeros((4, 4))
for (i, j), v in raw_thermal.items():
    NEW_B1_RAW[i - 1, j - 1] = v["b1"]
    NEW_B2_RAW[i - 1, j - 1] = v["b2"]

print("\n--- BIAS_RAW comparison ---")
print(f"{'link':>6s} | {'existing':>10s} | {'new (RAW)':>10s} | {'Δ [cm]':>8s}")
print("-" * 50)
for i in range(4):
    for j in range(4):
        if i == j: continue
        old = EXISTING_BIAS[i, j]
        new = NEW_BIAS_RAW[i, j]
        print(f"{i+1}->{j+1}   | {old:>+10.4f} | {new:>+10.4f} | {(new-old)*100:>+7.2f}")

print("\n--- THERMAL_B1 comparison ---")
print(f"{'link':>6s} | {'existing':>10s} | {'new (RAW)':>10s} | {'Δ [cm/°C]':>10s}")
print("-" * 54)
for i in range(4):
    for j in range(4):
        if i == j: continue
        old = EXISTING_B1[i, j]
        new = NEW_B1_RAW[i, j]
        print(f"{i+1}->{j+1}   | {old*100:>+10.3f} | {new*100:>+10.3f} | {(new-old)*100:>+10.3f}")

print("\n--- THERMAL_B2 comparison ---")
print(f"{'link':>6s} | {'existing':>10s} | {'new (RAW)':>10s} | {'Δ [cm/°C]':>10s}")
print("-" * 54)
for i in range(4):
    for j in range(4):
        if i == j: continue
        old = EXISTING_B2[i, j]
        new = NEW_B2_RAW[i, j]
        print(f"{i+1}->{j+1}   | {old*100:>+10.3f} | {new*100:>+10.3f} | {(new-old)*100:>+10.3f}")

# ------------------------------------------------------------
# 7. Per-link variance after RAW correction
# ------------------------------------------------------------
print("\n" + "=" * 90)
print(" POST-CORRECTION RESIDUAL VARIANCE  (on RAW, bias + thermal removed)")
print("=" * 90)

# Common reference
T_ref_avg = float(np.mean([v["T_avg_mean"]  for v in raw_thermal.values()]))
T_ref_diff = 0.0

print(f"  T_ref_avg  = {T_ref_avg:.3f} °C")
print(f"  T_ref_diff = {T_ref_diff:.3f} °C")

print(f"\n{'link':>6s} | {'variance (m²)':>14s} | {'σ (cm)':>8s}")
print("-" * 42)

R_var = np.zeros((4, 4))
for (i, j), gt in GT_PAIR.items():
    mask = (df["init_id"] == i) & (df["resp_id"] == j)
    g    = df[mask]
    if len(g) < 100: continue

    t  = raw_thermal[(i, j)]
    Ti = g["temp_uwb_init"].values
    Tj = g["temp_uwb_resp"].values

    T_avg  = 0.5 * (Ti + Tj)
    T_diff = Ti - Tj

    # Full correction
    corr = (raw_bias_dict[(i, j)]
            + t["b1"] * (T_avg  - t["T_avg_mean"])
            + t["b2"] * (T_diff - t["T_diff_mean"]))
    resid = g["dist_raw_m"].values - gt - corr

    var = float(np.var(resid))
    R_var[i - 1, j - 1] = var
    print(f"{i}->{j}   | {var:>14.6f} | {np.sqrt(var)*100:>8.3f}")

# ------------------------------------------------------------
# 8. Suggested new firmware tables
# ------------------------------------------------------------
print("\n" + "=" * 90)
print(" SUGGESTED FIRMWARE TABLES  (fit on dist_raw_m)")
print("=" * 90)

print("\nconstexpr float BIAS_RAW[4][4] = {")
for i in range(4):
    cells = ", ".join(
        f"{NEW_BIAS_RAW[i, j]:+10.5f}f" if i != j else "  0.00000f"
        for j in range(4)
    )
    print(f"    {{ {cells} }},   // R{i+1} outgoing")
print("};")

print("\nconstexpr float THERMAL_B1[4][4] = {")
for i in range(4):
    cells = ", ".join(
        f"{NEW_B1_RAW[i, j]:+10.5f}f" if i != j else " 0.00000f"
        for j in range(4)
    )
    print(f"    {{ {cells} }},   // R{i+1} outgoing")
print("};")

print("\nconstexpr float THERMAL_B2[4][4] = {")
for i in range(4):
    cells = ", ".join(
        f"{NEW_B2_RAW[i, j]:+10.5f}f" if i != j else " 0.00000f"
        for j in range(4)
    )
    print(f"    {{ {cells} }},   // R{i+1} outgoing")
print("};")

print(f"\nconstexpr float T_REF_AVG  = {T_ref_avg:.4f}f;")
print(f"constexpr float T_REF_DIFF = {T_ref_diff:.4f}f;")

Loaded 39387 rows
After status==1 filter: 39362 rows

 PER-LINK MEAN BIAS  (raw vs cfo)
  link |     n |      GT |   raw_mean   raw_bias |   cfo_mean   cfo_bias
------------------------------------------------------------------------------------------
1->2   |  3580 |  2.0000 |    21.2341   +19.2341 |     1.6049    -0.3951
1->3   |  3585 |  2.8284 |    24.4812   +21.6528 |     7.3546    +4.5262
1->4   |  3585 |  2.0000 |    21.5407   +19.5407 |     2.4879    +0.4879
2->1   |  2970 |  2.0000 |    60.5328   +58.5328 |    80.1892   +78.1892
2->3   |  2972 |  2.0000 |    43.0874   +41.0874 |    45.6097   +43.6097
2->4   |  2972 |  2.8284 |    42.2235   +39.3951 |    42.8375   +40.0091
3->1   |  3147 |  2.8284 |    58.5852   +55.7568 |    75.5588   +72.7304
3->2   |  3152 |  2.0000 |    38.0566   +36.0566 |    35.5488   +33.5488
3->4   |  3151 |  2.0000 |    39.0840   +37.0840 |    37.1889   +35.1889
4->1   |  3408 |  2.0000 |    59.6726   +57.6726 |    78.7314   +76.7314
4->2   |  3420 |  

In [2]:
# ============================================================
# Cell — Refit excluding the corrupted 4->3 link
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
np.set_printoptions(precision=4, suppress=True)

# ------------------------------------------------------------
# 1. Load (same as before)
# ------------------------------------------------------------
FILES = {
    1: "dataset_raw_static_square_r1_10min.csv",
    2: "dataset_raw_static_square_r2_10min.csv",
    3: "dataset_raw_static_square_r3_10min.csv",
    4: "dataset_raw_static_square_r4_10min.csv",
}

GT_PAIR = {
    (1, 2): 2.0000, (1, 3): 2.8284, (1, 4): 2.0000,
    (2, 1): 2.0000, (2, 3): 2.0000, (2, 4): 2.8284,
    (3, 1): 2.8284, (3, 2): 2.0000, (3, 4): 2.0000,
    (4, 1): 2.0000, (4, 2): 2.8284, (4, 3): 2.0000,
}

EXCLUDE_LINKS = {(4, 3)}   # corrupted link

frames = []
for rb, fname in FILES.items():
    p = Path(fname)
    if not p.exists():
        print(f"MISSING: {fname}")
        continue
    frames.append(pd.read_csv(p))

df = pd.concat(frames, ignore_index=True)
df = df[df["status"] == 1].copy()
df = df.dropna(subset=["dist_raw_m", "dist_cfo_m"]).reset_index(drop=True)
print(f"Loaded and filtered: {len(df)} rows")

# ------------------------------------------------------------
# 2. Investigate the 4->3 anomaly
# ------------------------------------------------------------
print("\n" + "=" * 80)
print(" INVESTIGATION: 4->3 LINK")
print("=" * 80)
g43 = df[(df["init_id"] == 4) & (df["resp_id"] == 3)]
print(f"  n              : {len(g43)}")
print(f"  dist_raw_m     : "
      f"mean={g43['dist_raw_m'].mean():.4f}  "
      f"std={g43['dist_raw_m'].std():.4f}  "
      f"min={g43['dist_raw_m'].min():.4f}  "
      f"max={g43['dist_raw_m'].max():.4f}")
print(f"  dist_cfo_m     : "
      f"mean={g43['dist_cfo_m'].mean():.4f}  "
      f"std={g43['dist_cfo_m'].std():.4f}  "
      f"min={g43['dist_cfo_m'].min():.4f}  "
      f"max={g43['dist_cfo_m'].max():.4f}")

# Show percentiles to reveal outliers
for col in ["dist_raw_m", "dist_cfo_m"]:
    print(f"\n  {col} percentiles:")
    for p in [1, 5, 25, 50, 75, 95, 99]:
        print(f"    p{p:>2d} = {np.percentile(g43[col], p):.4f}")

# Show sample rows near the extremes
print("\n  Sample rows with LARGEST dist_raw_m:")
print(g43.nlargest(5, "dist_raw_m")[
    ["frame_id", "dist_raw_m", "dist_cfo_m", "rssi_dbm",
     "std_noise", "fp_ampl1", "fp_ampl2", "lde_error"]
].to_string(index=False))

print("\n  Sample rows with SMALLEST dist_raw_m:")
print(g43.nsmallest(5, "dist_raw_m")[
    ["frame_id", "dist_raw_m", "dist_cfo_m", "rssi_dbm",
     "std_noise", "fp_ampl1", "fp_ampl2", "lde_error"]
].to_string(index=False))

# ------------------------------------------------------------
# 3. Per-link mean bias, excluding corrupted links
# ------------------------------------------------------------
print("\n" + "=" * 80)
print(" PER-LINK MEAN BIAS (raw only, excluding corrupted)")
print("=" * 80)
print(f"{'link':>6s} | {'n':>5s} | {'GT':>7s} | {'raw_mean':>10s} | {'raw_bias':>10s}")
print("-" * 62)

link_stats = {}
for (i, j), gt in GT_PAIR.items():
    if (i, j) in EXCLUDE_LINKS:
        continue
    mask = (df["init_id"] == i) & (df["resp_id"] == j)
    g = df[mask]
    if len(g) < 100:
        continue
    raw_mean = float(g["dist_raw_m"].mean())
    link_stats[(i, j)] = {
        "raw_mean": raw_mean,
        "raw_bias": raw_mean - gt,
        "n": len(g),
    }
    print(f"{i}->{j}   | {len(g):>5d} | {gt:>7.4f} | "
          f"{raw_mean:>10.4f} | {raw_mean - gt:>+10.4f}")

# ------------------------------------------------------------
# 4. Fit A + B + c model on clean data
# ------------------------------------------------------------
def fit_ABc(bias_dict):
    rows, y = [], []
    for (i, j), b in bias_dict.items():
        row = np.zeros(9)
        row[i - 1] = 1.0
        row[4 + j - 1] = 1.0
        row[8] = 1.0
        rows.append(row)
        y.append(b)
    A_mat = np.array(rows)
    y     = np.array(y)
    g1 = np.zeros(9); g1[0:4] = 1.0
    g2 = np.zeros(9); g2[4:8] = 1.0
    A_aug = np.vstack([A_mat, g1, g2])
    y_aug = np.concatenate([y, [0.0, 0.0]])
    coef, *_ = np.linalg.lstsq(A_aug, y_aug, rcond=None)
    A_vals = coef[0:4]
    B_vals = coef[4:8]
    c_val  = coef[8]
    pred   = A_mat @ coef
    resid  = y - pred
    return A_vals, B_vals, c_val, float(np.sqrt(np.mean(resid ** 2))), resid

raw_bias_dict = {k: v["raw_bias"] for k, v in link_stats.items()}

A, B, c, rmse, resid = fit_ABc(raw_bias_dict)

print("\n" + "=" * 80)
print(" CLEAN FIT  d_meas = d_true + A_i + B_j + c")
print("=" * 80)
print(f"  c       = {c:+.4f} m")
print(f"  RMSE    = {rmse*100:.2f} cm")
print(f"  A       = {['%+.4f' % a for a in A]}")
print(f"  B       = {['%+.4f' % b for b in B]}")
print(f"\n  Per-link residuals:")
for k, r in zip(raw_bias_dict.keys(), resid):
    print(f"    {k[0]}->{k[1]}: {r*100:+7.2f} cm")

# ------------------------------------------------------------
# 5. Implied directed-link bias matrix
# ------------------------------------------------------------
print("\n" + "=" * 80)
print(" CLEAN BIAS MATRIX  A_i + B_j + c")
print("=" * 80)
mat = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        if i == j: continue
        mat[i, j] = A[i] + B[j] + c

print("       " + "".join(f"  →R{j+1:<8d}" for j in range(4)))
for i in range(4):
    row_str = f"  R{i+1} |"
    for j in range(4):
        if i == j:
            row_str += f"  {'--':>9s}"
        else:
            row_str += f"  {mat[i,j]:+9.4f}"
    print(row_str)

# ------------------------------------------------------------
# 6. Thermal fit on clean data
# ------------------------------------------------------------
def fit_thermal(df, dist_col, bias_dict):
    out = {}
    for (i, j), gt in GT_PAIR.items():
        if (i, j) in EXCLUDE_LINKS: continue
        mask = (df["init_id"] == i) & (df["resp_id"] == j)
        g    = df[mask]
        if len(g) < 100: continue

        bias = g[dist_col].values - gt - bias_dict[(i, j)]

        Ti = g["temp_uwb_init"].values
        Tj = g["temp_uwb_resp"].values
        T_avg  = 0.5 * (Ti + Tj)
        T_diff = Ti - Tj
        T_avg_c  = T_avg  - T_avg.mean()
        T_diff_c = T_diff - T_diff.mean()

        X = np.column_stack([np.ones(len(bias)), T_avg_c, T_diff_c])
        beta, *_ = np.linalg.lstsq(X, bias, rcond=None)
        pred  = X @ beta
        resid = bias - pred
        ss_res = float(np.sum(resid ** 2))
        ss_tot = float(np.sum((bias - bias.mean()) ** 2))
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")

        out[(i, j)] = {
            "b1": beta[1], "b2": beta[2], "r2": r2,
            "rmse_cm": float(np.std(resid) * 100),
            "T_avg_mean":  float(T_avg.mean()),
            "T_diff_mean": float(T_diff.mean()),
        }
    return out

raw_thermal = fit_thermal(df, "dist_raw_m", raw_bias_dict)

print("\n" + "=" * 80)
print(" CLEAN THERMAL MODEL FIT")
print("=" * 80)
print(f"  {'link':>6s} | {'b1 [cm/°C]':>12s} {'b2 [cm/°C]':>12s} "
      f"{'R²':>8s} {'σ [cm]':>8s}")
print("  " + "-" * 60)
for k in sorted(raw_thermal.keys()):
    v = raw_thermal[k]
    print(f"  {k[0]}->{k[1]}   | {v['b1']*100:>+12.3f} "
          f"{v['b2']*100:>+12.3f} {v['r2']:>8.4f} {v['rmse_cm']:>8.3f}")

# ------------------------------------------------------------
# 7. Suggested firmware tables (clean)
# ------------------------------------------------------------
print("\n" + "=" * 80)
print(" SUGGESTED FIRMWARE TABLES  (clean, no 4->3)")
print("=" * 80)

print("\nconstexpr float BIAS_RAW[4][4] = {")
for i in range(4):
    cells = []
    for j in range(4):
        if i == j:
            cells.append("  0.00000f")
        elif (i+1, j+1) in EXCLUDE_LINKS:
            cells.append("  0.00000f")  # placeholder
        else:
            cells.append(f"{mat[i, j]:+10.5f}f")
    print(f"    {{ {', '.join(cells)} }},   // R{i+1} outgoing")
print("};")

print("\nconstexpr float THERMAL_B1[4][4] = {")
for i in range(4):
    cells = []
    for j in range(4):
        if i == j or (i+1, j+1) in EXCLUDE_LINKS:
            cells.append("  0.00000f")
        else:
            cells.append(f"{raw_thermal[(i+1, j+1)]['b1']:+10.5f}f")
    print(f"    {{ {', '.join(cells)} }},   // R{i+1} outgoing")
print("};")

print("\nconstexpr float THERMAL_B2[4][4] = {")
for i in range(4):
    cells = []
    for j in range(4):
        if i == j or (i+1, j+1) in EXCLUDE_LINKS:
            cells.append("  0.00000f")
        else:
            cells.append(f"{raw_thermal[(i+1, j+1)]['b2']:+10.5f}f")
    print(f"    {{ {', '.join(cells)} }},   // R{i+1} outgoing")
print("};")

# Common T_ref
T_avg_means = [v["T_avg_mean"] for v in raw_thermal.values()]
print(f"\nconstexpr float T_REF_AVG  = {np.mean(T_avg_means):.4f}f;")
print(f"constexpr float T_REF_DIFF = 0.0000f;")

Loaded and filtered: 39362 rows

 INVESTIGATION: 4->3 LINK
  n              : 3420
  dist_raw_m     : mean=23.8974  std=1106.2259  min=-64650.1250  max=43.1220
  dist_cfo_m     : mean=25.7570  std=1106.2226  min=-64648.0664  max=45.2055

  dist_raw_m percentiles:
    p 1 = 42.1039
    p 5 = 42.2634
    p25 = 42.6904
    p50 = 42.9062
    p75 = 43.0000
    p95 = 43.0610
    p99 = 43.0798

  dist_cfo_m percentiles:
    p 1 = 43.3340
    p 5 = 43.6398
    p25 = 44.4519
    p50 = 44.8411
    p75 = 45.0259
    p95 = 45.1338
    p99 = 45.1607

  Sample rows with LARGEST dist_raw_m:
 frame_id  dist_raw_m  dist_cfo_m  rssi_dbm  std_noise  fp_ampl1  fp_ampl2  lde_error
     3505     43.1220     45.2055    -76.43         16      1659      3694          0
     3503     43.1079     45.1998    -77.39         20      1556      3717          0
     3071     43.0985     45.1664    -74.17         16      1105      3398          0
     3223     43.0985     45.1653    -76.68         20      1614      314

In [1]:
# ============================================================
# Refit UWB model — all 12 links, remove only numerical outliers
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
np.set_printoptions(precision=4, suppress=True)

FILES = {
    1: "dataset_raw_static_square_r1_10min.csv",
    2: "dataset_raw_static_square_r2_10min.csv",
    3: "dataset_raw_static_square_r3_10min.csv",
    4: "dataset_raw_static_square_r4_10min.csv",
}

GT_PAIR = {
    (1, 2): 2.0000, (1, 3): 2.8284, (1, 4): 2.0000,
    (2, 1): 2.0000, (2, 3): 2.0000, (2, 4): 2.8284,
    (3, 1): 2.8284, (3, 2): 2.0000, (3, 4): 2.0000,
    (4, 1): 2.0000, (4, 2): 2.8284, (4, 3): 2.0000,
}

# Outlier thresholds on dist_raw_m
DIST_MIN = 0.0
DIST_MAX = 500.0     # safe: max valid raw is ~120 m for edge links

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------
frames = []
for rb, fname in FILES.items():
    p = Path(fname)
    if not p.exists():
        print(f"MISSING: {fname}")
        continue
    frames.append(pd.read_csv(p))

df = pd.concat(frames, ignore_index=True)
n0 = len(df)
df = df[df["status"] == 1].copy()
df = df.dropna(subset=["dist_raw_m", "dist_cfo_m"]).reset_index(drop=True)
n1 = len(df)

mask_valid = (df["dist_raw_m"] > DIST_MIN) & (df["dist_raw_m"] < DIST_MAX)
n_outliers = int((~mask_valid).sum())
df = df[mask_valid].reset_index(drop=True)

print(f"Total rows loaded       : {n0}")
print(f"After status==1 + dropna: {n1}")
print(f"Outliers removed        : {n_outliers}")
print(f"Final clean rows        : {len(df)}")

# ------------------------------------------------------------
# Per-link stats
# ------------------------------------------------------------
print("\n" + "=" * 80)
print(" PER-LINK MEAN BIAS (clean, all 12 links)")
print("=" * 80)
print(f"{'link':>6s} | {'n':>5s} | {'GT':>7s} | {'raw_mean':>10s} | {'raw_bias':>10s}")
print("-" * 62)

link_stats = {}
for (i, j), gt in GT_PAIR.items():
    mask = (df["init_id"] == i) & (df["resp_id"] == j)
    g = df[mask]
    if len(g) < 100:
        print(f"{i}->{j}   | {len(g):>5d} | (insufficient data)")
        continue
    raw_mean = float(g["dist_raw_m"].mean())
    link_stats[(i, j)] = {
        "raw_mean": raw_mean,
        "raw_bias": raw_mean - gt,
        "n": len(g),
    }
    print(f"{i}->{j}   | {len(g):>5d} | {gt:>7.4f} | "
          f"{raw_mean:>10.4f} | {raw_mean - gt:>+10.4f}")

# ------------------------------------------------------------
# Fit A + B + c
# ------------------------------------------------------------
def fit_ABc(bias_dict):
    rows, y = [], []
    for (i, j), b in bias_dict.items():
        row = np.zeros(9)
        row[i - 1] = 1.0
        row[4 + j - 1] = 1.0
        row[8] = 1.0
        rows.append(row)
        y.append(b)
    A_mat = np.array(rows); y = np.array(y)
    g1 = np.zeros(9); g1[0:4] = 1.0
    g2 = np.zeros(9); g2[4:8] = 1.0
    A_aug = np.vstack([A_mat, g1, g2])
    y_aug = np.concatenate([y, [0.0, 0.0]])
    coef, *_ = np.linalg.lstsq(A_aug, y_aug, rcond=None)
    A_vals = coef[0:4]; B_vals = coef[4:8]; c_val = coef[8]
    pred = A_mat @ coef
    resid = y - pred
    return A_vals, B_vals, c_val, float(np.sqrt(np.mean(resid ** 2))), resid

raw_bias_dict = {k: v["raw_bias"] for k, v in link_stats.items()}
A, B, c, rmse, resid = fit_ABc(raw_bias_dict)

print("\n" + "=" * 80)
print(" STATIC BIAS MODEL")
print("=" * 80)
print(f"  c    = {c:+.4f} m")
print(f"  RMSE = {rmse*100:.2f} cm")
print(f"  A    = {['%+.4f' % a for a in A]}")
print(f"  B    = {['%+.4f' % b for b in B]}")

mat = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        if i == j: continue
        mat[i, j] = A[i] + B[j] + c

print("\n" + "=" * 80)
print(" BIAS MATRIX  A_i + B_j + c")
print("=" * 80)
print("       " + "".join(f"  →R{j+1:<8d}" for j in range(4)))
for i in range(4):
    row_str = f"  R{i+1} |"
    for j in range(4):
        if i == j:
            row_str += f"  {'--':>9s}"
        else:
            row_str += f"  {mat[i,j]:+9.4f}"
    print(row_str)

# ------------------------------------------------------------
# Thermal fit
# ------------------------------------------------------------
def fit_thermal(df, dist_col, bias_dict):
    out = {}
    for (i, j), gt in GT_PAIR.items():
        mask = (df["init_id"] == i) & (df["resp_id"] == j)
        g = df[mask]
        if len(g) < 100: continue
        bias = g[dist_col].values - gt - bias_dict[(i, j)]
        Ti = g["temp_uwb_init"].values
        Tj = g["temp_uwb_resp"].values
        T_avg = 0.5 * (Ti + Tj); T_diff = Ti - Tj
        T_avg_c = T_avg - T_avg.mean(); T_diff_c = T_diff - T_diff.mean()
        X = np.column_stack([np.ones(len(bias)), T_avg_c, T_diff_c])
        beta, *_ = np.linalg.lstsq(X, bias, rcond=None)
        pred = X @ beta; resid = bias - pred
        ss_res = float(np.sum(resid ** 2))
        ss_tot = float(np.sum((bias - bias.mean()) ** 2))
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
        out[(i, j)] = {
            "b1": beta[1], "b2": beta[2], "r2": r2,
            "rmse_cm": float(np.std(resid) * 100),
            "T_avg_mean": float(T_avg.mean()),
            "T_diff_mean": float(T_diff.mean()),
        }
    return out

thermal = fit_thermal(df, "dist_raw_m", raw_bias_dict)

print("\n" + "=" * 80)
print(" THERMAL MODEL (on dist_raw_m)")
print("=" * 80)
print(f"  {'link':>6s} | {'b1 [cm/°C]':>12s} {'b2 [cm/°C]':>12s} "
      f"{'R²':>8s} {'σ [cm]':>8s}")
print("  " + "-" * 60)
for k in sorted(thermal.keys()):
    v = thermal[k]
    print(f"  {k[0]}->{k[1]}   | {v['b1']*100:>+12.3f} "
          f"{v['b2']*100:>+12.3f} {v['r2']:>8.4f} {v['rmse_cm']:>8.3f}")

# ------------------------------------------------------------
# Post-correction variance per link
# ------------------------------------------------------------
print("\n" + "=" * 80)
print(" POST-CORRECTION VARIANCE  (m²)")
print("=" * 80)
R_var = np.zeros((4, 4))
for (i, j), gt in GT_PAIR.items():
    mask = (df["init_id"] == i) & (df["resp_id"] == j)
    g = df[mask]
    if len(g) < 100: continue
    t = thermal[(i, j)]
    Ti = g["temp_uwb_init"].values
    Tj = g["temp_uwb_resp"].values
    T_avg = 0.5 * (Ti + Tj); T_diff = Ti - Tj
    corr = (raw_bias_dict[(i, j)]
            + t["b1"] * (T_avg - t["T_avg_mean"])
            + t["b2"] * (T_diff - t["T_diff_mean"]))
    resid = g["dist_raw_m"].values - gt - corr
    R_var[i-1, j-1] = float(np.var(resid))

print(f"{'link':>6s} | {'variance [m²]':>14s} | {'σ [cm]':>8s}")
print("-" * 42)
for i in range(4):
    for j in range(4):
        if i == j: continue
        v = R_var[i, j]
        print(f"{i+1}->{j+1}   | {v:>14.6f} | {np.sqrt(v)*100:>8.3f}")

# ------------------------------------------------------------
# Firmware tables
# ------------------------------------------------------------
print("\n" + "=" * 80)
print(" FIRMWARE TABLES")
print("=" * 80)

print("\nconstexpr float BIAS_RAW[4][4] = {")
for i in range(4):
    cells = []
    for j in range(4):
        if i == j: cells.append("  0.00000f")
        else: cells.append(f"{mat[i, j]:+10.5f}f")
    print(f"    {{ {', '.join(cells)} }},   // R{i+1} outgoing")
print("};")

print("\nconstexpr float THERMAL_B1[4][4] = {")
for i in range(4):
    cells = []
    for j in range(4):
        if i == j: cells.append("  0.00000f")
        else: cells.append(f"{thermal[(i+1, j+1)]['b1']:+10.5f}f")
    print(f"    {{ {', '.join(cells)} }},   // R{i+1} outgoing")
print("};")

print("\nconstexpr float THERMAL_B2[4][4] = {")
for i in range(4):
    cells = []
    for j in range(4):
        if i == j: cells.append("  0.00000f")
        else: cells.append(f"{thermal[(i+1, j+1)]['b2']:+10.5f}f")
    print(f"    {{ {', '.join(cells)} }},   // R{i+1} outgoing")
print("};")

print("\nconstexpr float VARIANCE_POST_THERMAL[4][4] = {")
for i in range(4):
    cells = []
    for j in range(4):
        if i == j: cells.append("  0.000000f")
        else: cells.append(f"{R_var[i, j]:>10.6f}f")
    print(f"    {{ {', '.join(cells)} }},   // R{i+1} outgoing")
print("};")

T_avg_means = [v["T_avg_mean"] for v in thermal.values()]
print(f"\nconstexpr float T_REF_AVG  = {np.mean(T_avg_means):.4f}f;")
print(f"constexpr float T_REF_DIFF = 0.0000f;")

Total rows loaded       : 39387
After status==1 + dropna: 39362
Outliers removed        : 1
Final clean rows        : 39361

 PER-LINK MEAN BIAS (clean, all 12 links)
  link |     n |      GT |   raw_mean |   raw_bias
--------------------------------------------------------------
1->2   |  3580 |  2.0000 |    21.2341 |   +19.2341
1->3   |  3585 |  2.8284 |    24.4812 |   +21.6528
1->4   |  3585 |  2.0000 |    21.5407 |   +19.5407
2->1   |  2970 |  2.0000 |    60.5328 |   +58.5328
2->3   |  2972 |  2.0000 |    43.0874 |   +41.0874
2->4   |  2972 |  2.8284 |    42.2235 |   +39.3951
3->1   |  3147 |  2.8284 |    58.5852 |   +55.7568
3->2   |  3152 |  2.0000 |    38.0566 |   +36.0566
3->4   |  3151 |  2.0000 |    39.0840 |   +37.0840
4->1   |  3408 |  2.0000 |    59.6726 |   +57.6726
4->2   |  3420 |  2.8284 |    41.0313 |   +38.2029
4->3   |  3419 |  2.0000 |    42.8134 |   +40.8134

 STATIC BIAS MODEL
  c    = +38.7524 m
  RMSE = 13.83 cm
  A    = ['-13.9730', '+5.6888', '+3.1521', '+5.1